## 1. `static_cast<type>(value)` - The Safe Default

### When to use:
- Converting between numeric types: `int` ↔ `double` ↔ `float`
- Casting `enum` to `int` and vice versa
- Upcasting/downcasting in class hierarchies (when you're sure it's safe)
- Converting pointers in related class hierarchies

### Characteristics:
- **Compile-time checked**: The compiler verifies the cast makes sense
- **Explicit and safe**: Clearly shows intent
- **Searchable**: Easy to find all casts in code

### Examples:
```cpp
// Numeric conversions
double pi = 3.14;
int rounded = static_cast<int>(pi); // 3

// Enum conversions
enum class Status { Ok = 200, NotFound = 404 };
Status s = static_cast<Status>(200);
int code = static_cast<int>(Status::NotFound); // 404

// Pointer conversions in class hierarchies
Base* base = new Derived();
Derived* derived = static_cast<Derived*>(base);
```

In [ ]:
#include <iostream>

enum class Status : uint8_t {
    Ok = 200,
    NotFound = 404,
    ServerError = 500
};

{
    // Example 1: Numeric conversion
    double temperature = 98.6;
    int temp_rounded = static_cast<int>(temperature);
    std::cout << "Temperature: " << temp_rounded << std::endl;
    
    // Example 2: Int to enum class
    int http_code = 404;
    Status status = static_cast<Status>(http_code);
    std::cout << "HTTP Status: " << static_cast<int>(status) << std::endl;
    
    // Example 3: Enum class to int
    Status response = Status::Ok;
    int code = static_cast<int>(response);
    std::cout << "Response code: " << code << std::endl;
}

## 2. `reinterpret_cast<type>(value)` - The Dangerous One

### When to use:
- Converting between completely unrelated pointer types
- Converting pointer to integer (and vice versa) for low-level operations
- System programming, hardware access, or bit manipulation

### Characteristics:
- **No compile-time checks**: Reinterprets bit pattern without conversion
- **Extremely dangerous**: Can cause undefined behavior
- **No type safety**: Bypasses all type safety mechanisms
- **Use rarely**: Only for low-level code with full understanding of consequences

### Examples:
```cpp
// Convert pointer to integer (addresses)
int* ptr = new int(42);
uintptr_t address = reinterpret_cast<uintptr_t>(ptr);

// Convert integer back to pointer (dangerous!)
int* ptr2 = reinterpret_cast<int*>(address);

// Convert between unrelated pointer types (dangerous!)
double* dptr = reinterpret_cast<double*>(ptr); // undefined behavior!
```

⚠️ **Warning:** Misuse of `reinterpret_cast` leads to crashes and security vulnerabilities.

In [ ]:
#include <iostream>
#include <cstdint>

{
    // Example 1: Pointer to integer (getting memory address)
    int x = 42;
    int* ptr = &x;
    uintptr_t address = reinterpret_cast<uintptr_t>(ptr);
    std::cout << "Address of x: 0x" << std::hex << address << std::endl;
    
    // Example 2: Integer back to pointer (unsafe!)
    // This is ONLY safe if the integer was a valid pointer address
    int* ptr2 = reinterpret_cast<int*>(address);
    std::cout << "Dereferenced: " << std::dec << *ptr2 << std::endl;
    
    // Example 3: Dangerous - converting between unrelated types
    // DO NOT DO THIS - just for demonstration!
    // double* wrong = reinterpret_cast<double*>(ptr); // undefined behavior
}

## 3. `const_cast<type>(value)` - The Const Remover

### When to use:
- Remove `const` from a pointer or reference
- Add `const` to a pointer or reference
- When interfacing with older C APIs that don't use `const` properly

### Characteristics:
- **Type-safe**: The underlying type remains unchanged
- **But risky**: Modifying a `const` object is undefined behavior
- **Last resort**: Usually indicates a design flaw if used frequently

### Examples:
```cpp
// Remove const from pointer
const int* cptr = new int(42);
int* ptr = const_cast<int*>(cptr);
*ptr = 100; // undefined behavior if original was truly const!

// Remove const from reference
const int& cref = some_const_value;
int& ref = const_cast<int&>(cref); // dangerous!
```

⚠️ **Warning:** Modifying a `const` object through `const_cast` is undefined behavior.

In [ ]:
#include <iostream>

// Legacy C function that doesn't use const
void legacy_function(char* str) {
    std::cout << "Legacy function: " << str << std::endl;
}

{
    // Example 1: Removing const to call legacy function
    const char* message = "Hello";
    // legacy_function(message); // ERROR: cannot pass const to non-const parameter
    legacy_function(const_cast<char*>(message)); // Unsafe but sometimes necessary
    
    // Example 2: Adding const
    int x = 42;
    int* ptr = &x;
    const int* cptr = const_cast<const int*>(ptr); // Adding const
    std::cout << "Value: " << *cptr << std::endl;
    
    // Example 3: const_cast with references
    const int& cref = x;
    int& ref = const_cast<int&>(cref);
    ref = 100; // This is safe here because x was not originally const
    std::cout << "Modified: " << x << std::endl;
}

## 4. `dynamic_cast<type>(value)` - Safe Polymorphic Casting

### When to use:
- Safely casting pointers/references in polymorphic class hierarchies
- Downcasting from base class to derived class with runtime checking
- When you need to check if an object is of a specific derived type

### Characteristics:
- **Runtime checking**: Verifies the cast is safe at runtime (requires RTTI - Run-Time Type Information)
- **Safe downcasting**: Returns `nullptr` for pointers (returns exception for references) if cast fails
- **Polymorphic classes only**: Only works with classes that have virtual functions
- **Compile-time checked**: Compiler verifies the classes are related
- **Performance cost**: Slower than `static_cast` due to runtime checks

### Examples:
```cpp
class Animal {
public:
    virtual ~Animal() = default;
    virtual void speak() const = 0;
};

class Dog : public Animal {
public:
    void speak() const override { std::cout << "Woof!" << std::endl; }
    void fetch() { std::cout << "Fetching..." << std::endl; }
};

// Safe downcasting
Animal* animal = new Dog();
Dog* dog = dynamic_cast<Dog*>(animal); // Returns non-null pointer if cast succeeds

if (dog != nullptr) {
    dog->fetch(); // Safe to call Dog-specific methods
} else {
    std::cout << "Not a Dog!" << std::endl;
}

// For references (throws exception on failure)
Dog& dog_ref = dynamic_cast<Dog&>(animal); // Throws std::bad_cast if fails
```

⚠️ **Requirements:** The base class must have at least one virtual function (like virtual destructor).


In [ ]:
#include <iostream>
#include <memory>

// Base class with virtual function (required for dynamic_cast)
class Shape {
public:
    virtual ~Shape() = default;
    virtual void draw() const = 0;
    virtual std::string getType() const = 0;
};

class Circle : public Shape {
private:
    double radius;
public:
    Circle(double r) : radius(r) {}
    void draw() const override { std::cout << "Drawing Circle" << std::endl; }
    std::string getType() const override { return "Circle"; }
    void calculateArea() { std::cout << "Area: " << 3.14 * radius * radius << std::endl; }
};

class Square : public Shape {
private:
    double side;
public:
    Square(double s) : side(s) {}
    void draw() const override { std::cout << "Drawing Square" << std::endl; }
    std::string getType() const override { return "Square"; }
    void calculatePerimeter() { std::cout << "Perimeter: " << 4 * side << std::endl; }
};

{
    // Example 1: Safe downcasting with pointer
    std::cout << "--- Example 1: Pointer downcasting ---" << std::endl;
    Shape* shape = new Circle(5.0);
    
    Circle* circle = dynamic_cast<Circle*>(shape);
    if (circle != nullptr) {
        std::cout << "Successfully cast to Circle!" << std::endl;
        circle->calculateArea();
    } else {
        std::cout << "Not a Circle" << std::endl;
    }
    
    // Example 2: Failed downcasting
    std::cout << "\n--- Example 2: Failed downcasting ---" << std::endl;
    Shape* shape2 = new Square(4.0);
    Circle* failed_cast = dynamic_cast<Circle*>(shape2);
    
    if (failed_cast == nullptr) {
        std::cout << "Cast failed - shape2 is not a Circle" << std::endl;
        std::cout << "Actual type: " << shape2->getType() << std::endl;
    }
    
    // Example 3: Reference downcasting (throws exception on failure)
    std::cout << "\n--- Example 3: Reference downcasting ---" << std::endl;
    try {
        Shape& shape_ref = *shape2;
        Square& square_ref = dynamic_cast<Square&>(shape_ref);
        square_ref.calculatePerimeter();
    } catch (const std::bad_cast& e) {
        std::cout << "Cast failed with exception: " << e.what() << std::endl;
    }
    
    delete shape;
    delete shape2;
}


## 4. Comparison Table

| Cast Type | Purpose | Compile-Time Check | Safety | Use Frequency |
|-----------|---------|-------------------|--------|---------------|
| `static_cast` | Type conversion | Yes ✓ | Safe | **Often** |
| `reinterpret_cast` | Bit reinterpretation | No ✗ | Dangerous | Rarely |
| `const_cast` | Remove/add const | Yes ✓ | Risky | Occasionally |
| `dynamic_cast` | Safe upcasting | Yes ✓ | Safe | Polymorphism |


## 5. Best Practices

✅ **DO:**
- Use `static_cast` by default for type conversions
- Use explicit C++ casts instead of C-style casts `(type)`
- Prefer avoiding casts altogether through good design
- Use `const_cast` only when interfacing with legacy code

❌ **DON'T:**
- Use `reinterpret_cast` unless absolutely necessary
- Use C-style casts in modern C++ code
- Modify `const` objects through `const_cast`
- Cast away type information without understanding consequences

### C-style casts to avoid:
```cpp
// OLD C-style (DON'T USE)
int x = (int)3.14;
Status s = (Status)404;

// NEW C++ style (USE THIS)
int x = static_cast<int>(3.14);
Status s = static_cast<Status>(404);
```